## 5.1 Layers and Blocks

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[-0.1248,  0.0591,  0.0387, -0.0867, -0.0757,  0.2253, -0.0354, -0.2502,
          0.0798,  0.0756],
        [-0.1608,  0.0272,  0.0659,  0.0246, -0.0692,  0.1417, -0.0075, -0.1665,
         -0.0235,  0.1146]], grad_fn=<AddmmBackward0>)

### 5.1.1. Self-defined Blocks

In [2]:
class MLP(nn.Module):
  # use model parameter to claim layers, here we use two FCL(fully connected layers)
  def __init__(self):
    # use Module, father class of MLP, to construct function to init
    # thus, while instantialize class one can also define other parameters, e.g. model params
    super().__init__()
    self.hidden = nn.Linear(20, 256) # hidden layer
    self.out = nn.Linear(256, 10) # output layer
    
  # define forward propagation, return model output with input x
  def forward(self, X):
    # hint, use ReLU, from nn.functional
    return self.out(F.relu(self.hidden(X)))

In [3]:
net = MLP()
net(X)

tensor([[-0.2058, -0.0303, -0.1164,  0.0669, -0.1115,  0.0048,  0.1177, -0.0777,
         -0.1278,  0.1170],
        [-0.1105, -0.0641, -0.1218,  0.0338, -0.1583, -0.0371,  0.0513, -0.0996,
         -0.1029,  0.0994]], grad_fn=<AddmmBackward0>)

### 5.1.2 Sequential Blocks

In [4]:
class MySequential(nn.Module):
  def __init__(self, *args):
    super().__init__()
    for idx, module in enumerate(args):
      # here, module is an instance in the subclass of Module, 
      # we save it in the member var _modules of class 'Module'
      # type of _module is OrderedDict
      self._modules[str(idx)] = module
  
  def forward(self, X):
    # OrderedDict guarantee tranverse with the order that member is added in
    for block in self._modules.values():
      X = block(X)
    return X

In [5]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.2660, -0.1184,  0.0186, -0.0162,  0.1653,  0.0785, -0.0980,  0.1807,
          0.1663,  0.0731],
        [-0.1083, -0.1823,  0.0191,  0.0651,  0.0785,  0.0415, -0.1642,  0.0744,
          0.1782,  0.0191]], grad_fn=<AddmmBackward0>)

### 5.1.3 Execute while Forwarding

In [6]:
class FixedHiddenMLP(nn.Module):
  def __init__(self):
    # rand weights that don't calculate gradient, so keep invariant while training
    super().__init__()
    self.rand_weight = torch.rand((20, 20), requires_grad=False)
    self.linear = nn.Linear(20, 20)
  
  def forward(self, X):
    X = self.linear(X)
    # use const params and relu, mm
    X = F.relu(torch.mm(X, self.rand_weight) + 1)
    # reuse FCL, equivalent to 2 FCLs sharing params
    X = self.linear(X)
    # control flow
    while X.abs().sum() > 1:
      X /= 2
    return X.sum()

In [7]:
net = FixedHiddenMLP()
net(X)

tensor(0.1847, grad_fn=<SumBackward0>)

In [8]:
class NestMLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential( nn.Linear(20, 64), nn.ReLU(),
                              nn.Linear(64, 32), nn.ReLU())
    self.linear = nn.Linear(32, 16)
  def forward(self, X):
    return self.linear(self.net(X))
  
chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(-0.2315, grad_fn=<SumBackward0>)